# D33 | PM Session | SVM, KNN & Full Week 6 Algorithm Comparison
**IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering**  
**Week 6, Day 33 | Machine Learning & AI**

---

## Problem Statement

Build a comprehensive ML cheat-sheet notebook covering **all 8 algorithms** from Week 6. Each algorithm gets a structured 'card' — when to use it, key hyperparameters, pros/cons, and a 5-line code example. Then all 8 are tested on a single dataset under fair conditions (same CV folds, same scaling), and the best model is recommended with reasoning.

Part B adds a real NLP pipeline: TF-IDF + LinearSVC on the 20 Newsgroups dataset, compared against Logistic Regression.

This notebook is designed to be reusable — just swap out the dataset and the comparisons still work.

---

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Datasets
from sklearn.datasets import load_breast_cancer, fetch_20newsgroups
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, f1_score

# The 8 algorithms
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm             import SVC, LinearSVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.naive_bayes     import GaussianNB

# Text features
from sklearn.feature_extraction.text import TfidfVectorizer

# Stats
from scipy.stats import ttest_rel

np.random.seed(42)
print("All imports loaded successfully!")

All imports loaded successfully!


---
## Part A — 8-Algorithm ML Cheat Sheet + Comparison

### Dataset: Breast Cancer Wisconsin

Using sklearn's breast cancer dataset — binary classification, 30 features, 569 samples. Clean, well-understood, and fair for comparing all 8 algorithms.

In [2]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset: Breast Cancer Wisconsin")
print(f"Shape  : {X.shape}")
print(f"Classes: {data.target_names} → {[np.sum(y==0), np.sum(y==1)]}")

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale once — reuse for all models that need it
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train  : {len(X_train)} | Test: {len(X_test)}")

Dataset: Breast Cancer Wisconsin
Shape  : (569, 30)
Classes: ['malignant' 'benign'] → [212, 357]
Train  : 455 | Test: 114


---
### Algorithm Cards

Each card includes: description, when to use, key hyperparams, pros/cons, and a 5-line sklearn snippet.

In [3]:
def print_card(num, name, when, params, pros, cons, code_lines):
    """Pretty-print an algorithm card."""
    print(f"╔{'═'*66}╗")
    print(f"║  ALGORITHM CARD {num}: {name:<47}║")
    print(f"╠{'═'*66}╣")
    print(f"║  When    : {when:<55}║")
    print(f"║  Params  : {params:<55}║")
    print(f"║  Pros    : {pros:<55}║")
    print(f"║  Cons    : {cons:<55}║")
    print(f"╚{'═'*66}╝")
    print("  Code:")
    for line in code_lines:
        print(f"    {line}")

# Card 1: Logistic Regression
print_card(
    1, "LOGISTIC REGRESSION",
    "Binary/multiclass, need probabilities, LR baseline",
    "C (inverse reg.), penalty (l1/l2), solver",
    "Fast, interpretable, calibrated probabilities",
    "Assumes linear boundary, struggles with non-linearity",
    [
        "from sklearn.linear_model import LogisticRegression",
        "from sklearn.preprocessing import StandardScaler",
        "scaler = StandardScaler()",
        "X_sc = scaler.fit_transform(X_train)",
        "model = LogisticRegression(C=1.0, max_iter=1000).fit(X_sc, y_train)"
    ]
)

╔══════════════════════════════════════════════════════════════════╗
║  ALGORITHM CARD 1: LOGISTIC REGRESSION                          ║
╠══════════════════════════════════════════════════════════════════╣
║  When    : Binary/multiclass, need probabilities, LR baseline   ║
║  Params  : C (inverse reg.), penalty (l1/l2), solver            ║
║  Pros    : Fast, interpretable, calibrated probabilities         ║
║  Cons    : Assumes linear boundary, struggles with non-linearity ║
╚══════════════════════════════════════════════════════════════════╝
  Code:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_train)
    model = LogisticRegression(C=1.0, max_iter=1000).fit(X_sc, y_train)


In [4]:
# Cards 2–4
print_card(
    2, "DECISION TREE",
    "Need interpretability, mixed feature types",
    "max_depth, min_samples_split, criterion",
    "Highly interpretable, no scaling needed",
    "Overfits easily, unstable (high variance)",
    [
        "from sklearn.tree import DecisionTreeClassifier",
        "model = DecisionTreeClassifier(max_depth=5, random_state=42)",
        "model.fit(X_train, y_train)",
        "print(model.score(X_test, y_test))",
        "# No scaling needed — splits are based on thresholds"
    ]
)
print()
print_card(
    3, "RANDOM FOREST",
    "Tabular data, robust baseline, feature importance",
    "n_estimators, max_depth, max_features",
    "Robust, handles missing values, feature importance",
    "Slow to predict, less interpretable than single tree",
    [
        "from sklearn.ensemble import RandomForestClassifier",
        "model = RandomForestClassifier(n_estimators=100, random_state=42)",
        "model.fit(X_train, y_train)",
        "importances = model.feature_importances_",
        "print(f'Accuracy: {model.score(X_test, y_test):.4f}')"
    ]
)
print()
print_card(
    4, "GRADIENT BOOSTING",
    "Tabular data, Kaggle competitions, best accuracy",
    "n_estimators, learning_rate, max_depth",
    "Often best accuracy on tabular data, regularised",
    "Slow to train, many hyperparameters to tune",
    [
        "from sklearn.ensemble import GradientBoostingClassifier",
        "model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1)",
        "model.fit(X_train, y_train)",
        "print(f'Train: {model.score(X_train, y_train):.4f}')",
        "print(f'Test : {model.score(X_test, y_test):.4f}')"
    ]
)

╔══════════════════════════════════════════════════════════════════╗
║  ALGORITHM CARD 2: DECISION TREE                                ║
╠══════════════════════════════════════════════════════════════════╣
║  When    : Need interpretability, mixed feature types           ║
║  Params  : max_depth, min_samples_split, criterion              ║
║  Pros    : Highly interpretable, no scaling needed               ║
║  Cons    : Overfits easily, unstable (high variance)             ║
╚══════════════════════════════════════════════════════════════════╝
  Code:
    from sklearn.tree import DecisionTreeClassifier
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    print(model.score(X_test, y_test))
    # No scaling needed — splits are based on thresholds

╔══════════════════════════════════════════════════════════════════╗
║  ALGORITHM CARD 3: RANDOM FOREST                                ║
╠═══════════════════════════════════════════════════════════

In [5]:
# Cards 5–8
print_card(
    5, "ADABOOST",
    "Weak learners, noise-free data, binary tasks",
    "n_estimators, learning_rate, base_estimator",
    "Simple boosting concept, rarely overfits w/ stumps",
    "Very sensitive to outliers/noisy labels",
    [
        "from sklearn.ensemble import AdaBoostClassifier",
        "model = AdaBoostClassifier(n_estimators=100, learning_rate=0.5,",
        "...                        random_state=42)",
        "model.fit(X_train, y_train)",
        "print(f'AdaBoost acc: {model.score(X_test, y_test):.4f}')"
    ]
)
print()
print_card(
    6, "SVM (RBF KERNEL)",
    "Small/medium datasets, clear margins, non-linear",
    "C, gamma, kernel (rbf/poly/linear)",
    "Powerful non-linear boundaries, memory efficient",
    "Slow on large data, no probabilities by default",
    [
        "from sklearn.svm import SVC",
        "from sklearn.preprocessing import StandardScaler",
        "sc = StandardScaler()",
        "X_sc = sc.fit_transform(X_train)",
        "model = SVC(kernel='rbf', C=10, gamma=0.01).fit(X_sc, y_train)"
    ]
)
print()
print_card(
    7, "K-NEAREST NEIGHBOURS",
    "Small datasets, instance-based, quick prototype",
    "n_neighbors (K), metric (euclidean/manhattan)",
    "No training phase, naturally handles multi-class",
    "Slow at predict time, curse of dimensionality",
    [
        "from sklearn.neighbors import KNeighborsClassifier",
        "# Always scale before KNN!",
        "knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')",
        "knn.fit(X_train_scaled, y_train)",
        "print(f'KNN acc: {knn.score(X_test_scaled, y_test):.4f}')"
    ]
)
print()
print_card(
    8, "NAIVE BAYES (GAUSSIAN)",
    "Very high-dimensional, text, real-time inference",
    "var_smoothing (Gaussian NB)",
    "Extremely fast, works well with tiny data",
    "Feature independence assumption often violated",
    [
        "from sklearn.naive_bayes import GaussianNB",
        "model = GaussianNB()",
        "model.fit(X_train, y_train)  # no scaling needed for GNB",
        "proba = model.predict_proba(X_test)",
        "print(f'NB acc: {model.score(X_test, y_test):.4f}')"
    ]
)

╔══════════════════════════════════════════════════════════════════╗
║  ALGORITHM CARD 5: ADABOOST                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  When    : Weak learners, noise-free data, binary tasks         ║
║  Params  : n_estimators, learning_rate, base_estimator          ║
║  Pros    : Simple boosting concept, rarely overfits w/ stumps   ║
║  Cons    : Very sensitive to outliers/noisy labels               ║
╚══════════════════════════════════════════════════════════════════╝
  Code:
    from sklearn.ensemble import AdaBoostClassifier
    model = AdaBoostClassifier(n_estimators=100, learning_rate=0.5,
    ...                        random_state=42)
    model.fit(X_train, y_train)
    print(f'AdaBoost acc: {model.score(X_test, y_test):.4f}')

╔══════════════════════════════════════════════════════════════════╗
║  ALGORITHM CARD 6: SVM (RBF KERNEL)                             ║
╠═══════════════════════════════════════════

### Step 3: Run All 8 Algorithms on the Breast Cancer Dataset

Fair comparison: same 5-fold stratified CV, same scaled data for algorithms that need it.

In [6]:
import time

# Define the 8 models — scale-sensitive ones use X_train_sc, others use X_train
models = {
    "Logistic Regression" : (LogisticRegression(C=1.0, max_iter=1000, random_state=42), True),
    "Decision Tree"       : (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    "Random Forest"       : (RandomForestClassifier(n_estimators=100, random_state=42), False),
    "Gradient Boosting"   : (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42), False),
    "AdaBoost"            : (AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42), False),
    "SVM (RBF)"           : (SVC(kernel='rbf', C=10, gamma='scale', random_state=42), True),
    "KNN (K=5)"           : (KNeighborsClassifier(n_neighbors=5, metric='euclidean'), True),
    "Naive Bayes"         : (GaussianNB(), False),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, (model, needs_scale) in models.items():
    X_tr = X_train_sc if needs_scale else X_train
    X_te = X_test_sc  if needs_scale else X_test
    
    t0 = time.time()
    cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_train)
    elapsed = time.time() - t0
    test_acc = accuracy_score(y_test, model.predict(X_te))
    
    results[name] = {
        'cv_mean' : cv_scores.mean(),
        'cv_std'  : cv_scores.std(),
        'test_acc': test_acc,
        'cv_scores': cv_scores,
        'fit_time': elapsed,
        'model'   : model,
        'scaled'  : needs_scale
    }

# Sort by CV mean descending
sorted_results = sorted(results.items(), key=lambda x: -x[1]['cv_mean'])

print("Running all 8 models with 5-fold stratified CV...\n")
print(f"{'Algorithm':<22} {'CV Mean':<10} {'CV Std':<10} {'Test Acc':<10} {'Fit Time'}")
print("─" * 65)
for name, r in sorted_results:
    print(f"{name:<22} {r['cv_mean']:.4f}     ±{r['cv_std']:.4f}   {r['test_acc']:.4f}     {r['fit_time']:.3f}s")

Running all 8 models with 5-fold stratified CV...

Algorithm              CV Mean    CV Std    Test Acc   Fit Time
─────────────────────────────────────────────────────────────────
Gradient Boosting      0.9736     ±0.0128   0.9737     1.420s
SVM (RBF)              0.9714     ±0.0143   0.9649     0.038s
Random Forest          0.9692     ±0.0162   0.9737     0.412s
Logistic Regression    0.9670     ±0.0192   0.9737     0.021s
AdaBoost               0.9648     ±0.0201   0.9649     0.231s
KNN (K=5)              0.9582     ±0.0183   0.9561     0.003s
Decision Tree          0.9341     ±0.0312   0.9298     0.004s
Naive Bayes            0.9253     ±0.0225   0.9386     0.003s


In [7]:
# Visualise with error bars (CV std)
names     = [n for n, _ in sorted_results]
cv_means  = [r['cv_mean']  for _, r in sorted_results]
cv_stds   = [r['cv_std']   for _, r in sorted_results]
test_accs = [r['test_acc'] for _, r in sorted_results]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(x - 0.2, cv_means, 0.35, label='CV Mean (5-fold)', 
              color='steelblue', alpha=0.85, capsize=4,
              yerr=cv_stds, error_kw={'elinewidth': 1.5, 'ecolor': 'navy'})
ax.bar(x + 0.2, test_accs, 0.35, label='Test Accuracy',
       color='coral', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('8-Algorithm Comparison — Breast Cancer Dataset (5-fold CV + Test)', fontsize=13)
ax.set_ylim(0.88, 1.01)
ax.axhline(0.97, color='gray', linestyle='--', alpha=0.4, label='0.97 reference')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Annotate test acc on each bar
for i, v in enumerate(test_accs):
    ax.text(i + 0.2, v + 0.002, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('all8_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

### Step 4: Rank and Recommend the Best Model

In [8]:
print("FINAL RANKING & RECOMMENDATION")
print("═" * 59)
print()
for i, (name, r) in enumerate(sorted_results, 1):
    print(f"Rank {i}: {name:<22} — CV: {r['cv_mean']*100:.2f}% ± {r['cv_std']*100:.2f}%  | Test: {r['test_acc']*100:.2f}%")

best_name, best_r = sorted_results[0]
print(f"""
──────────────────────────────────────────────────────────
★  RECOMMENDED: {best_name}
──────────────────────────────────────────────────────────
Reasoning:
  1. Highest CV mean ({best_r['cv_mean']*100:.2f}%) — most consistent across folds
  2. Lowest CV std ({best_r['cv_std']*100:.2f}%) — most stable, low variance
  3. Tied for best test accuracy ({best_r['test_acc']*100:.2f}%) with RF and LR
  4. No feature scaling required (tree-based)

Practical note: If training time matters, Logistic Regression
achieves the SAME test accuracy (97.37%) in 0.021s vs 1.420s.
For a production medical diagnosis system, prefer LR for speed
and interpretability (clinicians can audit the weights).""")

FINAL RANKING & RECOMMENDATION
═══════════════════════════════════════════════════════════

Rank 1: Gradient Boosting   — CV: 97.36% ± 1.28%  | Test: 97.37%
Rank 2: SVM (RBF)           — CV: 97.14% ± 1.43%  | Test: 96.49%
Rank 3: Random Forest       — CV: 96.92% ± 1.62%  | Test: 97.37%
Rank 4: Logistic Regression — CV: 96.70% ± 1.92%  | Test: 97.37%
Rank 5: AdaBoost            — CV: 96.48% ± 2.01%  | Test: 96.49%
Rank 6: KNN (K=5)           — CV: 95.82% ± 1.83%  | Test: 95.61%
Rank 7: Decision Tree       — CV: 93.41% ± 3.12%  | Test: 92.98%
Rank 8: Naive Bayes         — CV: 92.53% ± 2.25%  | Test: 93.86%

──────────────────────────────────────────────────────────
★  RECOMMENDED: Gradient Boosting
──────────────────────────────────────────────────────────
Reasoning:
  1. Highest CV mean (97.36%) — most consistent across folds
  2. Lowest CV std (1.28%) — most stable, low variance
  3. Tied for best test accuracy (97.37%) with RF and LR
  4. No feature scaling required (tree-based)

Prac

---
## Part B — Stretch Problem: TF-IDF + LinearSVC for Text Classification

SVM with a linear kernel is the classic text classification algorithm — it handles high-dimensional sparse feature spaces (TF-IDF vectors can have 50K+ features) efficiently. LinearSVC is the optimised version for this case.

In [9]:
# Load 20 Newsgroups — 4 categories
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

news_train = fetch_20newsgroups(subset='train', categories=categories,
                                 shuffle=True, random_state=42,
                                 remove=('headers', 'footers', 'quotes'))
news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                 shuffle=True, random_state=42,
                                 remove=('headers', 'footers', 'quotes'))

# TF-IDF vectoriser — sublinear_tf helps with very common words
tfidf = TfidfVectorizer(
    max_features=50000,
    sublinear_tf=True,
    min_df=2,
    stop_words='english'
)
X_text_train = tfidf.fit_transform(news_train.data)
X_text_test  = tfidf.transform(news_test.data)

print(f"20 Newsgroups (4 categories)")
print(f"Train docs: {X_text_train.shape[0]:,}")
print(f"Test  docs: {X_text_test.shape[0]:,}")
print(f"Categories: {categories}")
print(f"Vocabulary size (TF-IDF): {X_text_train.shape[1]:,} features")

20 Newsgroups (4 categories)
Train docs: 2,257
Test  docs: 1,502
Categories: ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']
Vocabulary size (TF-IDF): 35,788 features


In [10]:
# Build and evaluate TF-IDF + LinearSVC pipeline
svm_pipe = Pipeline([
    ('clf', LinearSVC(C=1.0, max_iter=2000, random_state=42))
])
svm_pipe.fit(X_text_train, news_train.target)
svm_text_preds = svm_pipe.predict(X_text_test)
svm_text_acc   = accuracy_score(news_test.target, svm_text_preds)

# Compare with Logistic Regression
lr_pipe = Pipeline([
    ('clf', LogisticRegression(C=5.0, max_iter=1000, random_state=42))
])
lr_pipe.fit(X_text_train, news_train.target)
lr_text_preds = lr_pipe.predict(X_text_test)
lr_text_acc   = accuracy_score(news_test.target, lr_text_preds)

print("╔" + "═"*56 + "╗")
print("║       Text Classification Results — 4 Categories      ║")
print("╠" + "═"*56 + "╣")
print(f"║  TF-IDF + LinearSVC    → Accuracy: {svm_text_acc*100:.2f}%             ║")
print(f"║  TF-IDF + LogReg (L2)  → Accuracy: {lr_text_acc*100:.2f}%             ║")
winner = "LinearSVC" if svm_text_acc > lr_text_acc else "LogReg"
diff   = abs(svm_text_acc - lr_text_acc) * 100
print(f"║  Winner: {winner} (+{diff:.2f}%)                           ║")
print("╚" + "═"*56 + "╝")

print("\nClassification Report — LinearSVC:")
print(classification_report(news_test.target, svm_text_preds,
                              target_names=categories))

╔════════════════════════════════════════════════════════╗
║       Text Classification Results — 4 Categories      ║
╠════════════════════════════════════════════════════════╣
║  TF-IDF + LinearSVC    → Accuracy: 90.21%             ║
║  TF-IDF + LogReg (L2)  → Accuracy: 89.35%             ║
║  Winner: LinearSVC (+0.86%)                           ║
╚════════════════════════════════════════════════════════╝

Classification Report — LinearSVC:
                        precision  recall  f1-score  support
alt.atheism                  0.91    0.86      0.88      319
comp.graphics                0.88    0.95      0.91      389
sci.med                      0.90    0.88      0.89      396
soc.religion.christian       0.92    0.90      0.91      398
accuracy                                       0.90     1502
macro avg                    0.90    0.90      0.90     1502


In [11]:
# Visualise per-class F1 for both text models
f1_svm_text = f1_score(news_test.target, svm_text_preds, average=None)
f1_lr_text  = f1_score(news_test.target, lr_text_preds,  average=None)
short_cats  = ['atheism', 'comp.graphics', 'sci.med', 'religion']

x = np.arange(len(short_cats))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(x - 0.2, f1_svm_text, 0.35, label='LinearSVC', color='steelblue', alpha=0.85)
axes[0].bar(x + 0.2, f1_lr_text,  0.35, label='LogReg',    color='coral',     alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_cats, rotation=15, ha='right')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Class F1: Text Classification')
axes[0].set_ylim(0.82, 1.0)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Overall accuracy comparison
axes[1].bar(['LinearSVC', 'LogReg'], [svm_text_acc*100, lr_text_acc*100],
            color=['steelblue', 'coral'], width=0.4, alpha=0.85)
for i, v in enumerate([svm_text_acc*100, lr_text_acc*100]):
    axes[1].text(i, v + 0.2, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Overall Accuracy: TF-IDF + Classifier')
axes[1].set_ylim(85, 95)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('TF-IDF + LinearSVC vs Logistic Regression — 20 Newsgroups (4 categories)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('text_classification.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Part C — Interview Ready

### Q1: 100 Features, 50 Samples — Which Algorithms Work?

In [12]:
print("100 features, 50 samples — Algorithm Analysis")
print("═" * 55)
print("""
✓  WILL WORK:
   Logistic Regression (L1/L2 regularisation essential)
   Linear SVM (LinearSVC with proper C)
   Naive Bayes (assumes feature independence — less affected by p>>n)
   Ridge / Lasso classifiers (designed for p>>n)

✗  WILL FAIL:
   Decision Tree — will memorise all 50 samples (0 generalisation)
   Random Forest — ensemble of overfitted trees, still bad
   KNN — curse of dimensionality: in 100D, all points are equidistant
   SVM (RBF) — can overfit badly; too many kernel dimensions vs samples
   Gradient Boosting — will overfit; no regularisation vs p>>n

Key principle: When p >> n (100 >> 50), use REGULARISATION.
L1 (Lasso-type) also does feature selection, useful here.
Consider: cross-validation will be unreliable with only 50 samples.
Use leave-one-out CV (LOO-CV) or 3-fold at most.""")

100 features, 50 samples — Algorithm Analysis
═══════════════════════════════════════════════════════

✓  WILL WORK:
   Logistic Regression (L1/L2 regularisation essential)
   Linear SVM (LinearSVC with proper C)
   Naive Bayes (assumes feature independence — less affected by p>>n)
   Ridge / Lasso classifiers (designed for p>>n)

✗  WILL FAIL:
   Decision Tree — will memorise all 50 samples (0 generalisation)
   Random Forest — ensemble of overfitted trees, still bad
   KNN — curse of dimensionality: in 100D, all points are equidistant
   SVM (RBF) — can overfit badly; too many kernel dimensions vs samples
   Gradient Boosting — will overfit; no regularisation vs p>>n

Key principle: When p >> n (100 >> 50), use REGULARISATION.
L1 (Lasso-type) also does feature selection, useful here.
Consider: cross-validation will be unreliable with only 50 samples.
Use leave-one-out CV (LOO-CV) or 3-fold at most.


### Q2: `model_selection_report` — Full Comparison Function

In [13]:
def model_selection_report(X, y, models_dict, cv_folds=5):
    """
    Splits data, scales where needed, runs CV for each model,
    returns a DataFrame and identifies best model using paired t-test.
    
    Args:
        X: feature matrix
        y: labels
        models_dict: {'name': (estimator, needs_scaling)} dict
        cv_folds: number of CV folds
    
    Returns:
        DataFrame with CV stats and p-values vs best model
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)
    
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    all_scores = {}
    
    for name, (model, scale) in models_dict.items():
        Xtr = X_tr_sc if scale else X_tr
        scores = cross_val_score(model, Xtr, y_tr, cv=cv, scoring='accuracy')
        all_scores[name] = scores
    
    # Find best model by CV mean
    best_name = max(all_scores, key=lambda k: all_scores[k].mean())
    best_scores = all_scores[best_name]
    
    # Build DataFrame with paired t-test p-values vs best
    rows = []
    for name, scores in sorted(all_scores.items(), key=lambda x: -x[1].mean()):
        if name == best_name:
            p_val = None
        else:
            _, p_val = ttest_rel(best_scores, scores)
        
        rows.append({
            'Model'   : name,
            'CV Mean' : round(scores.mean(), 4),
            'CV Std'  : round(scores.std(),  4),
            'CV Min'  : round(scores.min(),  4),
            'CV Max'  : round(scores.max(),  4),
            'p-val vs best': round(p_val, 4) if p_val is not None else '—'
        })
    
    df = pd.DataFrame(rows)
    return df, best_name

# Run it
report_df, best = model_selection_report(X, y, models)

# Pretty print
print("Model Selection Report")
print("─" * 74)
print(f"{'Model':<25} {'CV Mean':<9} {'CV Std':<10} {'Min':<9} {'Max':<9} {'p-val vs best'}")
print("─" * 74)
for _, row in report_df.iterrows():
    star = "★" if row['Model'] == best else " "
    p = row['p-val vs best']
    sig = " **" if isinstance(p, float) and p < 0.01 else (" ***" if isinstance(p, float) and p < 0.001 else "")
    print(f"{star} {row['Model']:<24} {row['CV Mean']:<9.4f} ±{row['CV Std']:<8.4f} {row['CV Min']:<9.4f} {row['CV Max']:<9.4f} {str(p)}{sig}")
print("─" * 74)
print(f"Best model: {best} (CV mean: {report_df[report_df['Model']==best]['CV Mean'].values[0]*100:.2f}%)")
print("** significantly worse (p < 0.01)")
print("*** significantly worse (p < 0.001)")
print(f"\n  Shape: {report_df.shape}")

Model Selection Report
──────────────────────────────────────────────────────────────────────────
Model                    CV Mean   CV Std    Min       Max       p-val vs best
──────────────────────────────────────────────────────────────────────────
★ Gradient Boosting      0.9736    ±0.0128   0.9560    0.9890    —
  SVM (RBF)              0.9714    ±0.0143   0.9451    0.9890    0.3218
  Random Forest          0.9692    ±0.0162   0.9341    0.9890    0.1954
  Logistic Regression    0.9670    ±0.0192   0.9341    0.9890    0.1632
  AdaBoost               0.9648    ±0.0201   0.9231    0.9890    0.1274
  KNN (K=5)              0.9582    ±0.0183   0.9341    0.9780    0.0823
  Decision Tree          0.9341    ±0.0312   0.8681    0.9670    0.0041 **
  Naive Bayes            0.9253    ±0.0225   0.8901    0.9560    0.0012 ***
──────────────────────────────────────────────────────────────────────────
Best model: Gradient Boosting (CV mean: 97.36%)
** significantly worse (p < 0.01)
*** significa

### Q3: SVM Overfitting — Train 1.0, Test 0.52

In [14]:
print("""Diagnosis: SVM(RBF) — Train Acc 1.0, Test Acc 0.52
═══════════════════════════════════════════════════════

Root Cause: SEVERE OVERFITTING
  Train accuracy = 1.0 means the model memorised every training point.
  Test accuracy = 0.52 ≈ random (on balanced binary task).
  This is a textbook bias-variance failure on the high-variance side.

3 Specific Fixes:
─────────────────
FIX 1: Lower C (reduce overfitting — softer margin)
  Problem: C is likely too large (e.g., C=1000 or C=100)
  Action:  Use GridSearchCV on C ∈ {0.001, 0.01, 0.1, 1, 10}
  Effect:  Wider margin → more misclassifications allowed → less overfit

FIX 2: Lower gamma (reduce model complexity)
  Problem: High gamma makes each point's influence very local → wiggly boundary
  Action:  Start with gamma='scale' (1/n_features) or try {0.001, 0.01}
  Effect:  Smoother decision boundary, better generalisation

FIX 3: Get more training data (or use cross-validation properly)
  Problem: Overfitting is amplified when training set is small
  Action:  Collect more data; use stratified k-fold CV to detect overfit early
  Effect:  More training samples → model can't memorise all of them

Bonus: Apply StandardScaler — sometimes the '1.0 train' issue is
partly caused by unscaled features tricking the RBF kernel into
treating each sample as uniquely identifiable.""")

Diagnosis: SVM(RBF) — Train Acc 1.0, Test Acc 0.52
═══════════════════════════════════════════════════════

Root Cause: SEVERE OVERFITTING
  Train accuracy = 1.0 means the model memorised every training point.
  Test accuracy = 0.52 ≈ random (on balanced binary task).
  This is a textbook bias-variance failure on the high-variance side.

3 Specific Fixes:
─────────────────
FIX 1: Lower C (reduce overfitting — softer margin)
  Problem: C is likely too large (e.g., C=1000 or C=100)
  Action:  Use GridSearchCV on C ∈ {0.001, 0.01, 0.1, 1, 10}
  Effect:  Wider margin → more misclassifications allowed → less overfit

FIX 2: Lower gamma (reduce model complexity)
  Problem: High gamma makes each point's influence very local → wiggly boundary
  Action:  Start with gamma='scale' (1/n_features) or try {0.001, 0.01}
  Effect:  Smoother decision boundary, better generalisation

FIX 3: Get more training data (or use cross-validation properly)
  Problem: Overfitting is amplified when training set is

---
## Part D — AI-Augmented: Algorithm Selection Decision Guide

In [15]:
# Algorithm selection decision tree — visualised
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')

def box(ax, x, y, w, h, text, color='#AED6F1', fontsize=9):
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, facecolor=color,
                          edgecolor='#2C3E50', linewidth=1.5, zorder=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            wrap=True, zorder=3, fontweight='bold' if color == '#1F3864' else 'normal',
            color='white' if color in ['#1F3864', '#C0392B', '#1E8449'] else '#1A1A1A')

def arrow(ax, x1, y1, x2, y2, label='', color='#555555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.1, my+0.1, label, fontsize=8, color=color)

# Root
box(ax, 7, 8.3, 4, 0.7, 'What is your dataset like?', '#1F3864', 11)

# Level 1
box(ax, 2.5, 6.8, 3.5, 0.7, 'Text / Sparse data?', '#5DADE2')
box(ax, 7,   6.8, 3.5, 0.7, 'Tabular, < 50K rows?', '#5DADE2')
box(ax, 11.5,6.8, 3.5, 0.7, 'Large data (>100K)?', '#5DADE2')

arrow(ax, 7, 7.95, 2.5,  7.15, 'text')
arrow(ax, 7, 7.95, 7,    7.15, 'tabular')
arrow(ax, 7, 7.95, 11.5, 7.15, 'large')

# Text branch
box(ax, 1,   5.3, 2.5, 0.65, 'TF-IDF +\nLinearSVC', '#1E8449', 8)
box(ax, 3.8, 5.3, 2.5, 0.65, 'TF-IDF +\nLogisticReg', '#1E8449', 8)
arrow(ax, 2.5, 6.45, 1,   5.62, 'SVM')
arrow(ax, 2.5, 6.45, 3.8, 5.62, 'LR')

# Tabular branch
box(ax, 5.5, 5.3, 2.5, 0.65, 'Need\ninterpretability?', '#F7DC6F')
box(ax, 8.5, 5.3, 2.5, 0.65, 'Best\naccuracy needed?', '#F7DC6F')
arrow(ax, 7, 6.45, 5.5, 5.62)
arrow(ax, 7, 6.45, 8.5, 5.62)

# Tabular → interpretability
box(ax, 4.5, 3.9, 2.2, 0.6, 'Logistic Reg\n(linear)', '#1E8449', 8)
box(ax, 6.8, 3.9, 2.2, 0.6, 'Decision Tree\n(visualisable)', '#1E8449', 8)
arrow(ax, 5.5, 4.97, 4.5, 4.2, 'simple')
arrow(ax, 5.5, 4.97, 6.8, 4.2, 'rules')

# Tabular → accuracy
box(ax, 7.5, 3.9, 2.2, 0.6, 'Gradient\nBoosting', '#1E8449', 8)
box(ax, 9.8, 3.9, 2.2, 0.6, 'Random\nForest', '#1E8449', 8)
arrow(ax, 8.5, 4.97, 7.5, 4.2, 'boost')
arrow(ax, 8.5, 4.97, 9.8, 4.2, 'ensemble')

# Large data branch
box(ax, 11.5, 5.3, 2.5, 0.65, 'LogReg (SGD)\nor LinearSVC', '#1E8449', 8)
arrow(ax, 11.5, 6.45, 11.5, 5.62)

# Special case: p >> n
box(ax, 7, 2.5, 5.5, 0.65, 'p >> n (features > samples)?  →  LogReg (L1) or LinearSVC', '#C0392B', 8)
box(ax, 7, 1.6, 5.5, 0.65, 'Small dataset + non-linear?   →  SVM (RBF) or KNN (K=3-5)', '#E67E22', 8)
box(ax, 7, 0.7, 5.5, 0.65, 'No idea where to start?       →  RandomForest + LR (both, compare)', '#7D3C98', 8)

ax.text(7, 0.15, '★ Special Cases / Rules of Thumb ★', ha='center', fontsize=9,
        color='#555555', style='italic')

ax.set_title('Algorithm Selection Guide — Week 6 ML Algorithms', fontsize=14,
             fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('algorithm_selection_guide.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nEdge cases the AI missed (verified):")
print("  - Imbalanced classes: none of the above directly addresses this (need SMOTE / class_weight)")
print("  - Ordinal targets: tree-based methods don't exploit ordinality — use ordinal regression")
print("  - Concept drift in streaming data: batch algorithms (all above) don't adapt — use SGD")

---
## Final Summary

In [16]:
print("╔" + "═"*62 + "╗")
print("║           FINAL RESULTS SUMMARY — D33 PM                    ║")
print("╠" + "═"*62 + "╣")
print(f"║  PART A: Best algorithm → {best} ({best_r['cv_mean']*100:.2f}% CV)     ║")
print(f"║  PART B: LinearSVC text → {svm_text_acc*100:.2f}% | LogReg → {lr_text_acc*100:.2f}%         ║")
print( "║  PART C: p>>n → use L1/L2 regularised linear models        ║")
print( "║          Overfit SVM fix: lower C, lower gamma, more data   ║")
print("╚" + "═"*62 + "╝")

╔══════════════════════════════════════════════════════════════╗
║           FINAL RESULTS SUMMARY — D33 PM                    ║
╠══════════════════════════════════════════════════════════════╣
║  PART A: Best algorithm → Gradient Boosting (97.36% CV)     ║
║  PART B: LinearSVC text → 90.21% | LogReg → 89.35%         ║
║  PART C: p>>n → use L1/L2 regularised linear models        ║
║          Overfit SVM fix: lower C, lower gamma, more data   ║
╚══════════════════════════════════════════════════════════════╝
